# Python Fundamentals Summary — ETF Strategy Project

**Name:** Jesse Wang
**Date:** 2026-08-19

Demonstrates Python, NumPy, and pandas on toy data, and shows the reusable helpers in `src/utils.py` that later pipeline stages import.

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')                    # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))     # so `from src....` imports work
print('working from:', ROOT.name)

working from: project


## 1) NumPy basics

Toy price matrix (5 days x 3 tickers) with a fixed seed so the numbers are reproducible.

In [2]:
import numpy as np

np.random.seed(7)
prices = 150 + np.random.randn(5, 3).cumsum(axis=0)
print('array shape:', prices.shape)
print('mean by column:', prices.mean(axis=0).round(3))
print('std  by column:', prices.std(axis=0).round(3))

array shape: (5, 3)
mean by column: [152.357 147.548 150.528]
std  by column: [0.531 1.355 0.424]


## 2) pandas with toy price data

Wide matrix -> tidy (date, ticker, close), the same shape the pipeline uses.

In [3]:
import pandas as pd

tickers = ['SPY', 'QQQ', 'GLD']
dates = pd.date_range('2026-01-01', periods=5, freq='D')
df = pd.DataFrame(prices, columns=tickers, index=dates).stack().rename('close').reset_index()
df.columns = ['date', 'ticker', 'close']
df['close'] = df['close'].round(2)
df

,date,ticker,close
0,2026-01-01,SPY,151.69
1,2026-01-01,QQQ,149.53
2,2026-01-01,GLD,150.03
3,2026-01-02,SPY,152.10
4,2026-01-02,QQQ,148.75
5,2026-01-02,GLD,150.03
6,2026-01-03,SPY,152.10
7,2026-01-03,QQQ,146.99
8,2026-01-03,GLD,151.05
9,2026-01-04,SPY,152.70


## 3) Reusable helpers from `src/utils.py`

The same functions the pipeline uses for summary stats and returns.

In [4]:
from src.utils import get_summary_stats, aggregate_by_category, compute_returns

print('summary stats (numeric columns):')
print(get_summary_stats(df))

print('\nmean close by ticker:')
print(aggregate_by_category(df, group_col='ticker', value_col='close'))

print('\ndaily returns (within each ticker):')
print(compute_returns(df, price_col='close', group_col='ticker', method='simple').tail(6))

summary stats (numeric columns):
                      date       close
count                   15   15.000000
mean   2026-01-03 00:00:00  150.143333
min    2026-01-01 00:00:00  146.100000
25%    2026-01-02 00:00:00  149.140000
50%    2026-01-03 00:00:00  150.640000
75%    2026-01-04 00:00:00  151.895000
max    2026-01-05 00:00:00  153.200000
std                    NaN    2.243870

mean close by ticker:
  ticker  count     mean     sum     min     max
0    GLD      5  150.526  752.63  150.03  151.05
1    QQQ      5  147.546  737.73  146.10  149.53
2    SPY      5  152.358  761.79  151.69  153.20

daily returns (within each ticker):
         date ticker   close    return
9  2026-01-04    SPY  152.70  0.003945
10 2026-01-04    QQQ  146.36 -0.004286
11 2026-01-04    GLD  150.88 -0.001125
12 2026-01-05    SPY  153.20  0.003274
13 2026-01-05    QQQ  146.10 -0.001776
14 2026-01-05    GLD  150.64 -0.001591
